# Step 7: KITTI data pipeline and sanity check

**Goal:** stand up the KITTI stage of the two-knob audit. This notebook builds the object-level dataset, trains the two per-view classifiers, and runs the KITTI analog of the Step 2 sanity check.

**What this notebook must prove before Step 8 is allowed to exist:**

1. **Knob 1 has signal:** per-sensor corruption degrades each view's own accuracy
2. **Knob 2a (semantic swap) is orthogonal:** each view's own accuracy stays flat while naive fusion collapses
3. **Knob 2b (geometric drift) is orthogonal and visible:** per-view accuracy is untouched by construction, while the spatial coupling signal (projection consistency) collapses

**Key terms (define upfront, use everywhere):**

- **T_gt:** the ground-truth extrinsic calibration shipped with KITTI. Used ONCE, at dataset construction time, to define which LiDAR points belong to each object. Frozen forever after.
- **T_used:** the runtime calibration consumed by fusion and by coupled reliability estimators. This is what Knob 2b perturbs.
- **Why the split matters:** if the views themselves consumed T_used, Knob 2b would corrupt the view inputs and Knobs 1 and 2 would no longer be orthogonal. Building views with T_gt only makes them calibration-free at runtime, so Knob 2b touches nothing but the coupling stage. This preserves the orthogonality the confound ratio depends on.

**Views (calibration-free at runtime):**

- **Camera view:** the 2D ground-truth box crop, featurized alone
- **LiDAR view:** points inside the 3D ground-truth box, canonicalized to a box-centered frame (shape only, pose removed), featurized alone


## 0. Data we need on disk

Download from the KITTI 3D object detection benchmark (free registration at cvlibs.net, "3D Object Detection Evaluation 2017"):

- **Left color images** (`data_object_image_2.zip`, ~12 GB)
- **Velodyne point clouds** (`data_object_velodyne.zip`, ~29 GB)
- **Camera calibration matrices** (`data_object_calib.zip`, ~16 MB)
- **Training labels** (`data_object_label_2.zip`, ~5 MB)

**Expected layout** after extraction (only `training/` is needed):

```
KITTI_ROOT/
  training/
    image_2/   000000.png ...
    velodyne/  000000.bin ...
    calib/     000000.txt ...
    label_2/   000000.txt ...
```

**Disk saver:** we do not need to extract all 7,481 frames. Set `EXTRACT_FROM_ZIPS = True` in the next cell, point `ZIP_DIR` at the downloaded zips, and the extraction cell pulls only the first `N_FRAMES` frames (plus the small calib and label sets in full).


In [ ]:
# ============ CONFIG ============
from pathlib import Path
import numpy as np, torch, matplotlib.pyplot as plt

SEED  = 0
rng   = np.random.default_rng(SEED)
torch.manual_seed(SEED)

KITTI_ROOT = Path("kitti")            # must contain training/{image_2, velodyne, calib, label_2}
N_FRAMES   = 200                       # pilot scale; raise later for the full run

EXTRACT_FROM_ZIPS = False              # True = pull a subset straight out of the downloaded zips
ZIP_DIR           = Path("downloads")  # where the 4 KITTI zips live (only if extracting)

CLASSES          = ["Car", "Pedestrian", "Cyclist"]
MIN_LIDAR_POINTS = 30                  # objects with fewer in-box points are dropped
MAX_TRUNCATION   = 0.5
MAX_OCCLUSION    = 2
MIN_CROP_PX      = 16                  # skip 2D boxes smaller than this on either side
CROP_SIZE        = 64                  # camera view: 64x64 RGB
N_POINTS         = 128                 # LiDAR view: points sampled per object
CAR_CAP_MULT     = 3                   # keep at most 3x the rarest class count of Cars

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device = {DEVICE}")


In [ ]:
# ============ OPTIONAL: extract only N_FRAMES from the zips ============
# Safe to skip if KITTI_ROOT is already populated.
if EXTRACT_FROM_ZIPS:
    import zipfile
    wanted = {f"{i:06d}" for i in range(N_FRAMES)}

    def extract_subset(zip_name, subdir, ext, full=False):
        zp = ZIP_DIR / zip_name
        assert zp.exists(), f"missing {zp}"
        with zipfile.ZipFile(zp) as z:
            members = [m for m in z.namelist()
                       if m.startswith(f"training/{subdir}/") and m.endswith(ext)
                       and (full or m.split("/")[-1].split(".")[0] in wanted)]
            for m in members:
                z.extract(m, KITTI_ROOT)
        print(f"{zip_name}: extracted {len(members)} files")

    extract_subset("data_object_image_2.zip",  "image_2",  ".png")
    extract_subset("data_object_velodyne.zip", "velodyne", ".bin")
    extract_subset("data_object_calib.zip",    "calib",    ".txt", full=True)
    extract_subset("data_object_label_2.zip",  "label_2",  ".txt", full=True)


## 1. KITTI coordinate frames in three sentences

**Intuition first.** The velodyne sits on the roof with x pointing forward; the camera sits behind the windshield with z pointing forward. `Tr_velo_to_cam` (a rigid rotation plus translation) moves points from the LiDAR frame into the camera frame, `R0_rect` applies a small rectification twist, and `P2` is the pinhole matrix that flattens 3D camera-frame points onto pixels.

**The one equation:**

```
pixel = P2 @ R0_rect @ Tr_velo_to_cam @ point_velo
```

**Where the knobs live:** Knob 2b perturbs `Tr_velo_to_cam` only (that is the physical mount, the thing that drifts with vibration and temperature). `P2` and `R0_rect` stay fixed.

**Labels:** each object gives a 2D image box, plus a 3D box as bottom-face center `(x, y, z)` in the rectified camera frame, dimensions `(h, w, l)`, and yaw `ry` around the camera y axis.


In [ ]:
# ============ GEOMETRY CORE (unit-tested) ============
import numpy as np


def parse_calib(calib_path):
    data = {}
    with open(calib_path) as f:
        for line in f:
            if ':' not in line:
                continue
            key, vals = line.split(':', 1)
            data[key.strip()] = np.array([float(v) for v in vals.split()])
    P2 = data['P2'].reshape(3, 4)
    R0 = np.eye(4)
    R0[:3, :3] = data['R0_rect'].reshape(3, 3)
    Tr = np.eye(4)
    Tr[:3, :4] = data['Tr_velo_to_cam'].reshape(3, 4)
    return {'P2': P2, 'R0_rect': R0, 'Tr_velo_cam': Tr}


def velo_to_rect(pts_velo, Tr_velo_cam, R0_rect):
    n = pts_velo.shape[0]
    hom = np.hstack([pts_velo, np.ones((n, 1))])          
    rect = (R0_rect @ Tr_velo_cam @ hom.T).T             
    return rect[:, :3]


def rect_to_image(pts_rect, P2):
    
    n = pts_rect.shape[0]
    hom = np.hstack([pts_rect, np.ones((n, 1))])          
    proj = (P2 @ hom.T).T                                 
    depth = proj[:, 2]
    uv = proj[:, :2] / np.clip(depth[:, None], 1e-6, None)
    return uv, depth


def rot_y(angle):
    c, s = np.cos(angle), np.sin(angle)
    return np.array([[c, 0, s], [0, 1, 0], [-c * 0 - s, 0, c]])


def points_in_3d_box(pts_rect, center, dims, ry, margin=0.0):
    h, w, l = dims
    p = pts_rect - np.asarray(center)                      
    c, s = np.cos(-ry), np.sin(-ry)
    x =  c * p[:, 0] + s * p[:, 2]
    z = -s * p[:, 0] + c * p[:, 2]
    y = p[:, 1]
    return (
        (np.abs(x) <= l / 2 + margin) &
        (np.abs(z) <= w / 2 + margin) &
        (y <= 0 + margin) & (y >= -h - margin)
    )


def canonicalize_points(pts_rect, center, dims, ry):
    h, _, _ = dims
    mid = np.asarray(center, dtype=float).copy()
    mid[1] -= h / 2                                       
    p = pts_rect - mid
    c, s = np.cos(-ry), np.sin(-ry)
    out = np.empty_like(p)
    out[:, 0] =  c * p[:, 0] + s * p[:, 2]
    out[:, 2] = -s * p[:, 0] + c * p[:, 2]
    out[:, 1] = p[:, 1]
    return out


# ---------- Knob 2b: extrinsic perturbation ----------

def small_rotation(axis, angle):
    
    axis = np.asarray(axis, dtype=float)
    axis = axis / np.linalg.norm(axis)
    K = np.array([[0, -axis[2], axis[1]],
                  [axis[2], 0, -axis[0]],
                  [-axis[1], axis[0], 0]])
    return np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * (K @ K)


def perturb_extrinsics(Tr_velo_cam, severity, rng,
                       max_rot_deg=2.0, max_trans_m=0.20):
    T = Tr_velo_cam.copy()
    if severity == 0:
        return T
    axis = rng.normal(size=3)
    dR = small_rotation(axis, np.deg2rad(max_rot_deg * severity))
    direc = rng.normal(size=3)
    direc = direc / np.linalg.norm(direc)
    dt = direc * (max_trans_m * severity)
    T[:3, :3] = dR @ T[:3, :3]
    T[:3, 3] = dR @ T[:3, 3] + dt
    return T


def projection_consistency(pts_velo, box2d, calib, Tr_used):
    if pts_velo.shape[0] == 0:
        return 0.0
    rect = velo_to_rect(pts_velo, Tr_used, calib['R0_rect'])
    uv, depth = rect_to_image(rect, calib['P2'])
    x1, y1, x2, y2 = box2d
    inside = (
        (depth > 0) &
        (uv[:, 0] >= x1) & (uv[:, 0] <= x2) &
        (uv[:, 1] >= y1) & (uv[:, 1] <= y2)
    )
    return float(inside.mean())


## 2. Build the object-level dataset

**One sample = one object** with two views and one label, mirroring the Handwritten setup so the Step 5/6 harness transfers directly.

**Per object we store:**

- `crop` - the 2D box image crop, resized to 64x64 (camera view input)
- `points` - `N_POINTS x 4` canonicalized in-box LiDAR points `(x, y, z, intensity)` (LiDAR view input)
- `pts_velo_raw` - the untouched in-box points in the velodyne frame, kept for Knob 2b (projection consistency needs raw geometry)
- `box2d`, `frame_id` - metadata the coupling stage will consume

**Filters:** truncation, occlusion, minimum in-box point count, minimum crop size. These remove degenerate objects, not hard ones.

**Class balance warning, read this:** at 200 frames expect very few Cyclists (roughly 30 to 50) and a flood of Cars. The cell below caps Cars and prints counts. If any class lands under ~40 samples, the honest fixes are (a) raise `N_FRAMES`, or (b) temporarily drop Cyclist and run 2-class. Do not silently trust a classifier trained on 25 cyclists.


In [ ]:
# ============ LABEL PARSING + EXTRACTION ============
from PIL import Image

def parse_labels(label_path):
    objs = []
    with open(label_path) as f:
        for line in f:
            p = line.split()
            if p[0] not in CLASSES:
                continue
            objs.append(dict(
                cls=p[0], truncated=float(p[1]), occluded=int(p[2]),
                box2d=tuple(float(v) for v in p[4:8]),
                dims=tuple(float(v) for v in p[8:11]),     # h, w, l
                loc=np.array([float(v) for v in p[11:14]]),
                ry=float(p[14]),
            ))
    return objs

def load_velo(bin_path):
    return np.fromfile(bin_path, dtype=np.float32).reshape(-1, 4)

def sample_points(pts, n, rng):
    if len(pts) >= n:
        idx = rng.choice(len(pts), n, replace=False)
    else:
        idx = rng.choice(len(pts), n, replace=True)
    return pts[idx]

frame_ids = sorted(p.stem for p in (KITTI_ROOT / "training/image_2").glob("*.png"))[:N_FRAMES]
assert frame_ids, f"no images found under {KITTI_ROOT}/training/image_2"
print(f"{len(frame_ids)} frames found")

samples, calibs, skipped = [], {}, 0
for fi, fid in enumerate(frame_ids):
    try:
        img   = Image.open(KITTI_ROOT / f"training/image_2/{fid}.png").convert("RGB")
        velo  = load_velo(KITTI_ROOT / f"training/velodyne/{fid}.bin")
        calib = parse_calib(KITTI_ROOT / f"training/calib/{fid}.txt")
        objs  = parse_labels(KITTI_ROOT / f"training/label_2/{fid}.txt")
    except FileNotFoundError:
        skipped += 1
        continue
    calibs[fid] = calib
    rect_all = velo_to_rect(velo[:, :3], calib["Tr_velo_cam"], calib["R0_rect"])  # T_gt, used ONCE
    W, H = img.size
    for obj in objs:
        if obj["truncated"] > MAX_TRUNCATION or obj["occluded"] > MAX_OCCLUSION:
            continue
        x1, y1, x2, y2 = obj["box2d"]
        x1, y1 = max(0, x1), max(0, y1); x2, y2 = min(W, x2), min(H, y2)
        if x2 - x1 < MIN_CROP_PX or y2 - y1 < MIN_CROP_PX:
            continue
        mask = points_in_3d_box(rect_all, obj["loc"], obj["dims"], obj["ry"], margin=0.05)
        if mask.sum() < MIN_LIDAR_POINTS:
            continue
        in_rect = rect_all[mask]
        in_int  = velo[mask, 3]
        canon   = canonicalize_points(in_rect, obj["loc"], obj["dims"], obj["ry"])
        pts4    = np.hstack([canon, in_int[:, None]]).astype(np.float32)
        crop = np.asarray(
            img.crop((x1, y1, x2, y2)).resize((CROP_SIZE, CROP_SIZE)), dtype=np.uint8)
        samples.append(dict(
            crop=crop,
            points=sample_points(pts4, N_POINTS, rng),
            pts_velo_raw=velo[mask, :3].copy(),   # velodyne frame, for Knob 2b
            box2d=(x1, y1, x2, y2),
            label=CLASSES.index(obj["cls"]),
            frame_id=fid,
        ))
    if (fi + 1) % 50 == 0:
        print(f"  {fi+1}/{len(frame_ids)} frames, {len(samples)} objects so far")

print(f"done: {len(samples)} objects, {skipped} frames skipped (missing files)")


In [ ]:
# ============ CLASS BALANCE + FRAME-LEVEL SPLIT ============
labels = np.array([s["label"] for s in samples])
counts = {c: int((labels == i).sum()) for i, c in enumerate(CLASSES)}
print("raw counts:", counts)

# cap Cars at CAR_CAP_MULT x the rarest non-Car class
rare = min(v for c, v in counts.items() if c != "Car")
car_idx  = np.where(labels == CLASSES.index("Car"))[0]
keep_car = set(rng.choice(car_idx, min(len(car_idx), CAR_CAP_MULT * rare), replace=False))
samples  = [s for i, s in enumerate(samples)
            if labels[i] != CLASSES.index("Car") or i in keep_car]
labels   = np.array([s["label"] for s in samples])
counts   = {c: int((labels == i).sum()) for i, c in enumerate(CLASSES)}
print("balanced counts:", counts)
for c, v in counts.items():
    if v < 40:
        print(f"WARNING: only {v} {c} samples. Raise N_FRAMES or drop this class before trusting results.")

# split by FRAME (no object from the same frame in both splits, prevents leakage)
uniq = sorted({s["frame_id"] for s in samples})
rng.shuffle(uniq)
n_tr = int(0.7 * len(uniq))
train_frames = set(uniq[:n_tr])
tr_idx = [i for i, s in enumerate(samples) if s["frame_id"] in train_frames]
te_idx = [i for i, s in enumerate(samples) if s["frame_id"] not in train_frames]
print(f"train objects: {len(tr_idx)}   test objects: {len(te_idx)}")
for name, idx in [("train", tr_idx), ("test", te_idx)]:
    ls = labels[idx]
    print(f"  {name}:", {c: int((ls == i).sum()) for i, c in enumerate(CLASSES)})


In [ ]:
# ============ SAVE PROCESSED DATASET FOR STEP 8 ============
torch.save(dict(
    samples=samples, calibs=calibs, classes=CLASSES,
    tr_idx=tr_idx, te_idx=te_idx, seed=SEED,
    n_frames=len(frame_ids), crop_size=CROP_SIZE, n_points=N_POINTS,
), "step7_kitti_objects.pt")
print("saved step7_kitti_objects.pt")


## 3. Per-view classifiers

**Design rule:** each classifier sees exactly one view and nothing else. No calibration, no other modality. This is what makes the views calibration-free and Knob 2b structurally unable to touch them.

- **Camera head:** small 3-block CNN on the 64x64 crop
- **LiDAR head:** PointNet-mini (shared per-point MLP, max pool, classifier) on the canonicalized point set

Both are deliberately small. The audit needs a reliability signal, not state of the art accuracy. Class weights handle the residual imbalance.


In [ ]:
# ============ MODELS ============
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

class ObjDataset(Dataset):
    def __init__(self, samples, idx):
        self.s = [samples[i] for i in idx]
    def __len__(self):
        return len(self.s)
    def __getitem__(self, i):
        s = self.s[i]
        crop = torch.from_numpy(s["crop"]).float().permute(2, 0, 1) / 255.0
        pts  = torch.from_numpy(s["points"]).float()          # (N_POINTS, 4)
        return crop, pts, s["label"]

class CropCNN(nn.Module):
    def __init__(self, n_cls):
        super().__init__()
        def block(ci, co):
            return nn.Sequential(nn.Conv2d(ci, co, 3, padding=1), nn.BatchNorm2d(co),
                                 nn.ReLU(), nn.MaxPool2d(2))
        self.net  = nn.Sequential(block(3, 32), block(32, 64), block(64, 128),
                                  nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.head = nn.Linear(128, n_cls)
    def forward(self, x):
        return self.head(self.net(x))

class PointNetMini(nn.Module):
    def __init__(self, n_cls):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Conv1d(4, 64, 1),  nn.BatchNorm1d(64),  nn.ReLU(),
            nn.Conv1d(64, 128, 1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 256, 1), nn.BatchNorm1d(256), nn.ReLU())
        self.head = nn.Sequential(nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, n_cls))
    def forward(self, x):                     # x: (B, N, 4)
        f = self.mlp(x.transpose(1, 2))       # (B, 256, N)
        return self.head(f.max(dim=2).values)

tr_ds, te_ds = ObjDataset(samples, tr_idx), ObjDataset(samples, te_idx)
tr_dl = DataLoader(tr_ds, batch_size=64, shuffle=True)
te_dl = DataLoader(te_ds, batch_size=128)

cw = torch.tensor([1.0 / max(counts[c], 1) for c in CLASSES], device=DEVICE)
cw = cw / cw.sum() * len(CLASSES)
print("class weights:", cw.cpu().numpy().round(2))


In [ ]:
# ============ TRAIN ============
def train(model, view, epochs=25, lr=1e-3):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for ep in range(epochs):
        model.train(); tot = 0.0
        for crop, pts, y in tr_dl:
            x = (crop if view == "cam" else pts).to(DEVICE)
            y = y.to(DEVICE)
            loss = F.cross_entropy(model(x), y, weight=cw)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(y)
        if (ep + 1) % 5 == 0:
            print(f"  epoch {ep+1:2d}  loss {tot/len(tr_ds):.4f}")
    return model

@torch.no_grad()
def eval_probs(model, view, crops=None, points=None, loader=None):
    """Return softmax probs. Either pass a loader, or explicit crop/point tensors."""
    model.eval(); out = []
    if loader is not None:
        for crop, pts, _ in loader:
            x = (crop if view == "cam" else pts).to(DEVICE)
            out.append(F.softmax(model(x), dim=1).cpu())
    else:
        x = (crops if view == "cam" else points)
        for i in range(0, len(x), 256):
            out.append(F.softmax(model(x[i:i+256].to(DEVICE)), dim=1).cpu())
    return torch.cat(out)

print("training camera head")
cam_net = train(CropCNN(len(CLASSES)), "cam")
print("training LiDAR head")
lid_net = train(PointNetMini(len(CLASSES)), "lid")

y_te   = torch.tensor([samples[i]["label"] for i in te_idx])
p_cam  = eval_probs(cam_net, "cam", loader=te_dl)
p_lid  = eval_probs(lid_net, "lid", loader=te_dl)
acc = lambda p, y: (p.argmax(1) == y).float().mean().item()
print(f"\ncamera view acc: {acc(p_cam, y_te):.3f}")
print(f"LiDAR  view acc: {acc(p_lid, y_te):.3f}")
print(f"naive fused acc: {acc((p_cam + p_lid) / 2, y_te):.3f}")

torch.save(dict(cam=cam_net.state_dict(), lid=lid_net.state_dict()), "step7_kitti_nets.pt")


## 4. Sanity check: the three claims

Same logic as Step 2 on Handwritten, adapted to two knobs. All evaluations below run on the test split only.

| Claim | Intervention | Must observe |
|---|---|---|
| **Knob 1 has signal** | corrupt one sensor's input | that view's own accuracy drops |
| **Knob 2a is orthogonal** | mis-pair crops and point sets | each view's own accuracy flat, fused accuracy collapses |
| **Knob 2b is orthogonal and visible** | perturb `T_used` | per-view accuracy untouched by construction, projection consistency collapses |

**Reading the Knob 2a numbers:** "own accuracy" means each head scored against the label of ITS OWN sample. The fused prediction is scored against the camera sample's label (either choice collapses, that is the point).


In [ ]:
# ============ CLAIM 1: KNOB 1 HAS SIGNAL ============
te_crops  = torch.stack([torch.from_numpy(samples[i]["crop"]).float().permute(2,0,1)/255.0
                         for i in te_idx])
te_points = torch.stack([torch.from_numpy(samples[i]["points"]).float() for i in te_idx])

def corrupt_image(crops, severity, rng):
    """Gaussian pixel noise, std scales with severity (images in [0,1])."""
    noise = torch.from_numpy(rng.normal(0, 0.30 * severity, crops.shape)).float()
    return (crops + noise).clamp(0, 1)

def corrupt_lidar(points, severity, rng):
    """Coordinate jitter + point dropout, both scale with severity."""
    pts = points.clone()
    pts[..., :3] += torch.from_numpy(rng.normal(0, 0.10 * severity, pts[..., :3].shape)).float()
    n_drop = int(0.80 * severity * pts.shape[1])
    if n_drop > 0:
        for b in range(pts.shape[0]):          # drop then duplicate survivors
            keep = rng.choice(pts.shape[1], pts.shape[1] - n_drop, replace=False)
            refill = rng.choice(keep, n_drop, replace=True)
            pts[b] = pts[b][np.concatenate([keep, refill])]
    return pts

r1 = np.random.default_rng(SEED + 1)
for sev in [0.0, 0.5, 1.0]:
    a_cam = acc(eval_probs(cam_net, "cam", crops=corrupt_image(te_crops, sev, r1)), y_te)
    a_lid = acc(eval_probs(lid_net, "lid", points=corrupt_lidar(te_points, sev, r1)), y_te)
    print(f"severity {sev:.1f}   camera acc {a_cam:.3f}   LiDAR acc {a_lid:.3f}")
print("\nEXPECT: both columns fall as severity rises. If a view barely moves, its")
print("corruption is too weak to serve as Knob 1; strengthen it before Step 8.")


In [ ]:
# ============ CLAIM 2: KNOB 2A IS ORTHOGONAL ============
r2 = np.random.default_rng(SEED + 2)
n = len(te_idx)
for sev in [0.0, 0.5, 1.0]:
    perm = np.arange(n)
    n_mis = int(sev * n)
    if n_mis > 1:
        sub = r2.choice(n, n_mis, replace=False)
        perm[sub] = sub[np.argsort(r2.random(n_mis))]      # shuffle a subset of pairings
    p_lid_m = p_lid[perm]                                   # LiDAR view of the WRONG sample
    y_lid_m = y_te[perm]
    fused   = (p_cam + p_lid_m) / 2
    print(f"severity {sev:.1f}   cam-own {acc(p_cam, y_te):.3f}   "
          f"lid-own {acc(p_lid_m, y_lid_m):.3f}   fused {acc(fused, y_te):.3f}")
print("\nEXPECT: cam-own and lid-own flat across rows (pristine inputs), fused falls.")
print("Note: with 3 classes fused accuracy under full mispairing floors near the")
print("majority-class rate, not near 0.1 as on 10-class Handwritten.")


In [ ]:
# ============ CLAIM 3: KNOB 2B IS ORTHOGONAL AND VISIBLE ============
print("per-view accuracy under Knob 2b: unchanged by construction (views never consume T_used)\n")

r3 = np.random.default_rng(SEED + 3)
severities = [0.0, 0.25, 0.5, 0.75, 1.0]
curves = []
for sev in severities:
    vals = []
    for i in te_idx:
        s = samples[i]
        calib  = calibs[s["frame_id"]]
        T_used = perturb_extrinsics(calib["Tr_velo_cam"], sev, r3)
        vals.append(projection_consistency(s["pts_velo_raw"], s["box2d"], calib, T_used))
    curves.append(np.mean(vals))
    print(f"severity {sev:.2f}   mean projection consistency {curves[-1]:.3f}")

plt.figure(figsize=(6, 4))
plt.plot(severities, curves, "o-")
plt.xlabel("Knob 2b severity (extrinsic drift, 0 to 2 deg + 0 to 0.2 m)")
plt.ylabel("mean projection consistency")
plt.title("Spatial coupling signal collapses under pure misalignment")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print("\nEXPECT: a clear monotone drop. Both sensors are pristine at every point on")
print("this curve. Any reliability estimator built on this signal will blame a")
print("sensor for a mounting problem. That is the Masquerade on real geometry.")


## 5. Verdict and handoff to Step 8

**If all three claims held**, the KITTI pipeline is sound and the knobs are orthogonal on real sensor data. Record the numbers in the campaign log.

**What Step 8 (`step8_kitti_panel.ipynb`) will add:**

- The full reliability panel through the shared harness: TMC on the two views (isolated), deep ensembles (isolated), projection consistency as the coupled spatial method, optionally the JS-divergence coupled baseline for continuity with Step 6
- Both knobs on the shared [0,1] severity grid, fitted slopes S_q and S_a, and the confound ratio per method **per knob** (S_a reported separately for 2a and 2b)
- The taxonomy figure: isolated methods immune to both misalignments, decision-level coupling fooled by 2a, spatial coupling fooled by 2b

**Artifacts saved by this notebook:**

- `step7_kitti_objects.pt` - processed dataset with raw velodyne points and calibs (Step 8 input)
- `step7_kitti_nets.pt` - trained per-view heads
